# Figure 9 — ARD as a free sensitivity analysis

One length-scale per input dimension. A large fitted length-scale means the model barely uses that factor. Here the landscape is synthetic so the answer is known in advance and ARD can be checked. The last cell shows the three-line change needed to run the same analysis on the real EDBO direct-arylation dataset.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

## A 5-factor surface where we decide in advance which factors matter

Factors 1 and 2 drive the response strongly, factor 3 weakly, and factors 4 and 5 not at all. If ARD works, the fitted inverse length-scales should recover that ordering.

**Inputs are scaled to [0,1] before fitting.** Length-scales carry the units of their input, so comparing a length-scale in °C against one in mol% is meaningless unless you normalise first.

In [ ]:

rng = np.random.default_rng(0)
D = 5
NAMES = ["temperature", "catalyst loading", "residence time", "stir rate", "vial lot"]
TRUE_W = np.array([1.0, 0.85, 0.30, 0.0, 0.0])


def f5(U):
    """U in [0,1]^5. Only the first three columns do anything."""
    U = np.atleast_2d(U)
    z = (0.95 * np.exp(-((U[:, 0] - 0.65) ** 2) / 0.06)
         + 0.80 * np.exp(-((U[:, 1] - 0.40) ** 2) / 0.08)
         + 0.55 * U[:, 0] * U[:, 1]
         + 0.22 * np.sin(3.0 * U[:, 2]))
    return 100.0 * z / 2.2


N = 160
U = rng.random((N, D))
y = f5(U) + rng.normal(0, 1.5, N)
print("n =", N, " yield range", round(y.min(), 1), "-", round(y.max(), 1))

## Fit hyperparameters by maximising the log marginal likelihood

Random restarts on a log grid — crude, transparent, and fast enough at this size. A real package uses gradients.

In [ ]:

(ls, sf, sn), lml = gpmod.fit_hypers_ard(U, y, D, kernel=gpmod.matern52,
                                         seed=1, n_restarts=250)
print("fitted length-scales :", np.round(ls, 3))
print("fitted output scale  :", round(sf, 3))
print("fitted noise sd      :", round(sn, 3))
print("log marginal lik.    :", round(lml, 2))
relevance = 1.0 / ls
relevance = relevance / relevance.max()
for n_, r_, t_ in zip(NAMES, relevance, TRUE_W):
    print(f"  {n_:18s} ARD relevance {r_:5.2f}   (built in: {t_:.2f})")

In [ ]:

order = np.argsort(relevance)
fig, ax = plt.subplots(figsize=(6.0, 3.2))
cols = [style.RED if relevance[i] > 0.5 else
        (style.GOLD if relevance[i] > 0.2 else style.GRAY) for i in order]
ax.barh(np.arange(D), relevance[order], color=cols, height=0.62)
ax.plot(TRUE_W[order] / TRUE_W.max(), np.arange(D), "o", ms=6,
        color=style.INK, mfc="white", mew=1.4, label="built into the surface")
ax.set_yticks(np.arange(D))
ax.set_yticklabels([NAMES[i] for i in order])
ax.set_xlabel(r"ARD relevance,  $(1/\ell)$ normalised   —   inputs scaled to [0,1]")
ax.set_xlim(0, 1.12)
ax.legend(loc="lower right", fontsize=8.8)
ax.set_title("the campaign tells you which factors mattered", loc="left",
             fontsize=10.5)
for i, k in enumerate(order):
    ax.text(relevance[k] + 0.02, i, f"{relevance[k]:.2f}", va="center",
            fontsize=9, color=style.GRAY)
style.save(fig, "fig_09_ard_relevance", OUT)

## The same analysis on the real dataset

`_shared/edbo_data.py` downloads the 1,728-experiment direct-arylation dataset (reaction 3 of Shields et al. 2021). One-hot the three categorical columns, scale the two continuous ones to [0,1], and call the same fitter. Runs only if there is network access.

In [ ]:

import edbo_data
if edbo_data.available():
    df = edbo_data.load()
    print(df.shape, "rows loaded")
    cats = ["Base_SMILES", "Ligand_SMILES", "Solvent_SMILES"]
    Xr = [df[c].astype("category").cat.codes.to_numpy()[:, None] for c in cats]
    Xr = np.hstack(Xr).astype(float)
    Xr = Xr / np.maximum(Xr.max(0), 1)
    con = df[["Concentration", "Temp_C"]].to_numpy(float)
    con = (con - con.min(0)) / (con.max(0) - con.min(0))
    Ur = np.hstack([Xr, con])
    yr = df["yield"].to_numpy(float)
    sub = np.random.default_rng(0).choice(len(Ur), 400, replace=False)
    (lsr, sfr, snr), lmlr = gpmod.fit_hypers_ard(Ur[sub], yr[sub], Ur.shape[1],
                                                 seed=2, n_restarts=250)
    rel = (1 / lsr) / (1 / lsr).max()
    for n_, r_ in zip(["base", "ligand", "solvent", "concentration",
                       "temperature"], rel):
        print(f"  {n_:14s} ARD relevance {r_:5.2f}")
else:
    print("No network — skipping the real-data fit.")
    print("Run this cell on a connected machine to reproduce it.")